# Comportamiento bajo carga uniaxial — El ensayo de tracción

*Teoría de la Plasticidad — Bloque I, Sección 1. Cuaderno interactivo.*

El **comportamiento plástico** se caracteriza por:
* deformación **permanente** (irreversible), y
* deformación que **depende de la historia de carga** $\;\Rightarrow\;$ se describe mediante
  una teoría *incremental*.

En todo el Bloque I suponemos un **sólido plástico ideal**:
* isotropía,
* sin histéresis elástica,
* **sin efecto Bauschinger** (mismo límite elástico en tracción y compresión),
* independencia del tiempo.

El **ensayo de tracción simple** es el experimento fundamental para caracterizar un material
y calibrar todos los modelos de plasticidad posteriores. Este cuaderno lo construye de forma
interactiva:

1. **§1.1** tensión y deformación ingenieriles,
2. **§1.2** tensión y deformación verdaderas,
3. **§1.3** inestabilidad en tracción y estricción (Considère),
4. **§1.4** leyes empíricas tensión–deformación.

Los deslizadores actualizan la figura **en tiempo real**. Ejecuta todas las celdas de arriba
abajo (**Kernel ▸ Restart Kernel and Run All Cells**). Cada figura tiene una barra de
herramientas (lupa para **zoom**, mano para desplazar, casita para reiniciar la vista).

In [ ]:
%matplotlib widget
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import FloatSlider
from IPython.display import display

# --- Tipografía estilo LaTeX: Computer Modern en las fórmulas ($...$) y texto serif
#     con acentos (DejaVu Serif) para el español. No requiere instalar TeX; funciona
#     en Binder tal cual. (No se usa 'cmr10' porque carece de vocales acentuadas.)
plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ['Latin Modern Roman', 'CMU Serif', 'DejaVu Serif'],
    'mathtext.fontset': 'cm',
    'mathtext.rm': 'serif',
    'axes.formatter.use_mathtext': True,
    'axes.unicode_minus': False,
    'figure.dpi': 110,
    'font.size': 12,
})

OFFSET = 0.002   # desfase convencional de deformación plástica (0,2%) del límite elástico


def panel(update, sliders, fig):
    # Muestra los deslizadores sobre una figura PERSISTENTE.
    # La figura se crea una sola vez; al mover un deslizador solo se redibuja su
    # contenido (no se recrea el lienzo), por lo que la respuesta es inmediata.
    fig.canvas.header_visible = False          # oculta el titulillo 'Figure n'
    out = widgets.interactive_output(update, sliders)
    display(widgets.VBox(list(sliders.values())), fig.canvas, out)

def fix_limits(store, ax, xlim, ylim):
    # Congela los ejes en la PRIMERA vista (la de los valores por defecto): los
    # deslizadores cambian la curva, no el encuadre. Si la curva se sale del
    # cuadro, es intencionado.
    if not store:
        store['x'], store['y'] = xlim, ylim
    ax.set_xlim(*store['x']); ax.set_ylim(*store['y'])

## 1.1 El ensayo de tracción simple

Una probeta de longitud inicial $L_0$ y sección inicial $A_0$ se somete a una fuerza $F$,
alargándose $\Delta L$.

**Tensión ingenieril** (referida a la sección *inicial*):

$$S=\frac{F}{A_0}\qquad\left[\tfrac{\mathrm N}{\mathrm m^2}=\mathrm{Pa}\right]$$

**Deformación ingenieril** (adimensional):

$$e=\frac{\Delta L}{L_0}$$

La curva $S\!-\!e$ presenta una región **elástica** (pendiente $=E$), una región **plástica**
y finalmente **estricción** (*necking*) hasta la rotura. Las propiedades que se obtienen del
ensayo son:

* el **módulo de elasticidad** $E$,
* la **resistencia a tracción** $S_\max$ (el pico de la curva ingenieril, UTS),
* el **límite elástico convencional** $S_Y=F_x/A_0$, definido con un desfase de deformación
  plástica $x$ (habitualmente $x=0{,}2\%=0{,}002$).

## 1.2 Tensión y deformación verdaderas

A diferencia de la tensión ingenieril (referida a $A_0$), la **tensión verdadera** se refiere
a la sección *instantánea* $A$:

$$\sigma=\frac{F}{A}.$$

La deformación ingenieril **no es aditiva**. La **deformación verdadera** se define de forma
incremental y **sí** es aditiva:

$$\mathrm d\varepsilon=\frac{\mathrm dL}{L}\;\Rightarrow\;
\varepsilon=\int_{L_0}^{L}\frac{\mathrm dL}{L}=\ln\frac{L}{L_0}.$$

Suponiendo **incompresibilidad plástica** ($V=A_0L_0=AL=\text{cte}$) se obtiene la conversión
entre magnitudes verdaderas e ingenieriles:

$$\boxed{\;\varepsilon=\ln(1+e)\qquad\sigma=S\,(1+e)\;}$$

Válidas mientras la deformación es **uniforme** (hasta la estricción). La celda inferior
representa ambas curvas para el mismo material y muestra cómo divergen al crecer la deformación.

In [ ]:
def flow_curve(E, Y, n, eps):
    """Tramo elástico y endurecimiento potencial (tipo Hollomon), tensión-deformación
    verdaderas. Continua en la fluencia: sigma(eps_y)=Y, con K = Y / eps_y**n."""
    eps_y = Y/E
    K = Y/eps_y**n
    sig = np.where(eps <= eps_y, E*eps, K*eps**n)
    return sig, eps_y, K


# --- figura PERSISTENTE (se crea una sola vez) ---
plt.ioff()
_lim1 = {}
fig1, ax1 = plt.subplots(figsize=(8.2, 5.6))
plt.ion()


def plot_eng_vs_true(E_GPa=200.0, Y=250.0, n=0.15, eps_max=0.45):
    ax = ax1
    ax.clear()                                         # solo se redibuja el contenido

    E = E_GPa*1.0e3
    eps = np.linspace(1e-6, eps_max, 400)
    sig, eps_y, K = flow_curve(E, Y, n, eps)          # magnitudes VERDADERAS

    e = np.expm1(eps)                                  # deformación ingenieril  e = exp(eps)-1
    S = sig*np.exp(-eps)                               # tensión ingenieril  S = sigma/(1+e)

    eps_neck = n                                       # Considère: estricción en eps = n
    sig_neck = K*eps_neck**n
    S_max = sig_neck*np.exp(-eps_neck)                 # UTS (tensión ingenieril máxima)
    e_neck = np.expm1(eps_neck)

    ax.plot(eps, sig, color='crimson', lw=2.2, label=r'verdadera  $\sigma\,(\varepsilon)$')

    m = e <= e_neck                                    # tramo uniforme (válido)
    ax.plot(e[m],  S[m],  color='steelblue', lw=2.2, label=r'ingenieril  $S\,(e)$')
    ax.plot(e[~m], S[~m], color='steelblue', lw=1.6, ls='--', alpha=0.5,
            label=r'ing. (tras estricción, no uniforme)')

    # construcción del desfase del 0,2% para el límite elástico
    off = E*(eps - OFFSET)
    ax.plot(eps[off <= Y*1.05], off[off <= Y*1.05], color='0.6', lw=1.0, ls=':')
    ax.plot(eps_y, Y, 'o', color='k', ms=6)
    ax.annotate(r'$S_Y=%.0f$' % Y, (eps_y, Y), textcoords='offset points',
                xytext=(8, -12), fontsize=11)

    # estricción / UTS
    ax.plot(e_neck, S_max, 's', color='steelblue', ms=9, zorder=5)
    ax.annotate(r'$S_{\max}=%.0f$' % S_max, (e_neck, S_max),
                textcoords='offset points', xytext=(8, 6), fontsize=11, color='steelblue')
    ax.plot(eps_neck, sig_neck, 'o', color='crimson', ms=8, zorder=5)
    ax.axvline(eps_neck, color='0.7', lw=0.8, ls=':')
    ax.annotate(r'estricción  $\varepsilon=n=%.2f$' % n, (eps_neck, 0),
                textcoords='offset points', xytext=(6, 8), fontsize=10, color='0.35')

    fix_limits(_lim1, ax, (0, eps_max), (0, max(sig.max(), S_max)*1.1))
    ax.set_xlabel(r'deformación  $\varepsilon,\;e$  (-)')
    ax.set_ylabel(r'tensión  $\sigma,\;S$  (MPa)')
    ax.grid(alpha=0.3)
    ax.legend(loc='lower right', fontsize=10)
    ax.set_title(r'Curva ingenieril vs. verdadera:  $\sigma=S(1+e)$,  $\varepsilon=\ln(1+e)$',
                 fontsize=12)
    fig1.canvas.draw_idle()                            # refresco inmediato


panel(plot_eng_vs_true, dict(
    E_GPa=FloatSlider(value=200, min=50, max=400, step=10,
                      description='E (GPa)', style={'description_width':'initial'}),
    Y=FloatSlider(value=250, min=50, max=600, step=10,
                  description='límite elást. Y (MPa)', style={'description_width':'initial'}),
    n=FloatSlider(value=0.15, min=0.02, max=0.5, step=0.01,
                  description='exp. endurec. n', style={'description_width':'initial'}),
), fig1)

## 1.3 Inestabilidad en tracción y estricción

La carga es $F=\sigma A$. Durante el ensayo $\sigma$ **crece** (endurecimiento) mientras $A$
**decrece** (incompresibilidad). La carga alcanza un **máximo**; más allá, la deformación se
localiza en un **cuello** (estricción) y la tensión ingenieril cae. Imponiendo $\mathrm dF=0$:

$$\mathrm dF=\mathrm d(\sigma A)=0
\;\Rightarrow\;\frac{\mathrm d\sigma}{\sigma}=-\frac{\mathrm dA}{A}=\mathrm d\varepsilon
\;\Rightarrow\;\boxed{\dfrac{\mathrm d\sigma}{\mathrm d\varepsilon}=\sigma}\quad\text{(Considère)}$$

Para una ley de Hollomon $\sigma=K\varepsilon^{\,n}$ se obtiene la **deformación uniforme**

$$nK\varepsilon^{\,n-1}=K\varepsilon^{\,n}\;\Rightarrow\;\varepsilon_{\text{estr}}=n,$$

es decir, la deformación verdadera en la estricción coincide con el exponente de endurecimiento.
La estricción es un fenómeno **estructural** (depende de la geometría *y* del material), no solo
del material.

**Construcción de Considère**: en el punto de estricción la tangente a $\sigma(\varepsilon)$
tiene *subtangente igual a 1*, por lo que la recta tangente corta $\sigma=0$ en
$\varepsilon=n-1$.

In [ ]:
# --- figura PERSISTENTE (dos paneles) ---
plt.ioff()
_lim2L, _lim2R = {}, {}
fig2, (ax2L, ax2R) = plt.subplots(1, 2, figsize=(12.0, 5.0), constrained_layout=True)
plt.ion()


def plot_considere(K=520.0, n=0.15, eps_max=0.5):
    axL, axR = ax2L, ax2R
    axL.clear(); axR.clear()

    eps = np.linspace(1e-4, eps_max, 400)
    sig = K*eps**n                       # curva verdadera de Hollomon (flujo plástico)

    eps_neck = n
    sig_neck = K*eps_neck**n
    S = sig*np.exp(-eps)                 # tensión ingenieril
    e = np.expm1(eps)
    S_max = sig_neck*np.exp(-eps_neck)
    e_neck = np.expm1(eps_neck)

    # --- IZQDA: construcción de la tangente de Considère ---
    axL.plot(eps, sig, color='crimson', lw=2.2, label=r'$\sigma=K\varepsilon^{\,n}$')
    xt = np.array([eps_neck - 1.0, min(eps_max, eps_neck + 0.15)])
    axL.plot(xt, sig_neck*(1.0 + (xt - eps_neck)), color='0.4', lw=1.3, ls='--',
             label='tangente (subtangente $=1$)')
    axL.plot(eps_neck, sig_neck, 'o', color='crimson', ms=9, zorder=5)
    axL.plot(eps_neck - 1.0, 0.0, 'o', color='0.4', ms=5)
    axL.axvline(eps_neck, color='0.7', lw=0.8, ls=':')
    fix_limits(_lim2L, axL, (eps_neck - 1.05, eps_max), (0, sig.max()*1.1))
    axL.set_xlabel(r'deformación verdadera  $\varepsilon$  (-)')
    axL.set_ylabel(r'tensión verdadera  $\sigma$  (MPa)')
    axL.grid(alpha=0.3); axL.legend(loc='upper left', fontsize=10)
    axL.set_title(r'Construcción de Considère:  $\varepsilon_{\rm estr}=n=%.2f$' % n, fontsize=12)

    # --- DCHA: la carga (tensión ingenieril) tiene un máximo en la estricción ---
    axR.plot(e, S, color='steelblue', lw=2.2, label=r'ingenieril  $S\,(e)$')
    axR.plot(e_neck, S_max, 's', color='steelblue', ms=10, zorder=5)
    axR.annotate(r'$S_{\max}$ (UTS)', (e_neck, S_max), textcoords='offset points',
                 xytext=(8, -4), fontsize=11, color='steelblue')
    axR.axvline(e_neck, color='0.7', lw=0.8, ls=':')
    fix_limits(_lim2R, axR, (0, np.expm1(eps_max)), (0, S.max()*1.2))
    axR.set_xlabel(r'deformación ingenieril  $e$  (-)')
    axR.set_ylabel(r'tensión ingenieril  $S$  (MPa)')
    axR.grid(alpha=0.3); axR.legend(loc='lower right', fontsize=10)
    axR.set_title(r'Carga máxima $\;\Rightarrow\;$ inicio de estricción', fontsize=12)

    fig2.canvas.draw_idle()


panel(plot_considere, dict(
    K=FloatSlider(value=520, min=200, max=1200, step=20,
                  description='resistencia K (MPa)', style={'description_width':'initial'}),
    n=FloatSlider(value=0.15, min=0.02, max=0.5, step=0.01,
                  description='exp. endurec. n', style={'description_width':'initial'}),
), fig2)

## 1.4 Leyes empíricas tensión–deformación

Para representar analíticamente la parte plástica de la curva $\sigma\!-\!\varepsilon$ se
recurre a leyes empíricas de ajuste. Con $\varepsilon_y=Y/E$ (deformación de fluencia) y
$\varepsilon_p\simeq\varepsilon-\varepsilon_y$ (deformación plástica), las leyes que se
comparan abajo son:

**1. Elástico–perfectamente plástico**

$$\sigma=\begin{cases}E\,\varepsilon & \varepsilon\le\varepsilon_y\\[4pt] Y & \varepsilon>\varepsilon_y\end{cases}$$

**2. Elástico–lineal con endurecimiento** ($H$ = módulo de endurecimiento)

$$\sigma=\begin{cases}E\,\varepsilon & \varepsilon\le\varepsilon_y\\[4pt] Y+H\,(\varepsilon-\varepsilon_y) & \varepsilon>\varepsilon_y\end{cases}$$

**3. Hollomon (ley potencial)**

$$\sigma=K_H\,\varepsilon_p^{\,n}\qquad(0\le n\le 1)$$

$K_H$ es el **coeficiente de resistencia** y $n$ el **exponente de endurecimiento por
deformación** ($n=0$ $\Rightarrow$ perfectamente plástico). Ambos se obtienen **ajustando la
zona plástica**: en ejes $\log\sigma$ frente a $\log\varepsilon_p$ la ley es una recta de
pendiente $n$ y ordenada en el origen $\log K_H$, de donde $K_H=\sigma(\varepsilon_p=1)$.

> **Ojo:** Hollomon **no tiene término de límite elástico**: $\sigma\to 0$ cuando
> $\varepsilon_p\to 0$. Solo es válida en la zona plástica, por eso abajo se dibuja
> **únicamente a partir de la fluencia** y *no empalma* con la recta elástica. Esa carencia
> es justamente lo que corrige Ludwik.

**4. Ludwik**

$$\sigma=\sigma_Y+K_L\,\varepsilon_p^{\,n_L}$$

Añade el término aditivo $\sigma_Y$, de modo que la rama plástica **arranca en el límite
elástico** ($\sigma=\sigma_Y$ cuando $\varepsilon_p=0$).

**5. Ramberg–Osgood**

$$\varepsilon=\frac{\sigma}{E}+\alpha\left(\frac{\sigma}{Y}\right)^{m},
\qquad\alpha=0{,}002$$

Representa de forma **continua** la transición elástico-plástica, sin un límite elástico
definido de forma abrupta.

---

La celda inferior las superpone todas para el mismo $E$ y $Y$. **Cada ley tiene sus propios
parámetros**: $K_H,\,n$ (Hollomon) y $K_L,\,n_L$ (Ludwik) se controlan por separado, junto
con $H$ y el exponente $m$ de Ramberg–Osgood.

In [ ]:
# --- figura PERSISTENTE ---
plt.ioff()
_lim3 = {}
fig3, ax3 = plt.subplots(figsize=(8.2, 5.6))
plt.ion()


def plot_laws(E_GPa=200.0, Y=250.0, H=1500.0,
              K_H=650.0, n=0.15, K_L=500.0, n_L=0.15, m=8.0, eps_max=0.05):
    ax = ax3
    ax.clear()

    E = E_GPa*1.0e3
    eps_y = Y/E                              # deformación de fluencia
    eps = np.linspace(1e-6, eps_max, 400)
    epsp = np.clip(eps - eps_y, 0.0, None)   # deformación plástica (aprox.)
    plast = eps > eps_y                      # zona plástica

    # 1) elástico - perfectamente plástico
    epp = np.where(eps <= eps_y, E*eps, Y)
    # 2) elástico - lineal con endurecimiento
    lin = np.where(eps <= eps_y, E*eps, Y + H*(eps - eps_y))
    # 3) Hollomon: sigma = K_H * eps_p^n. Sin término de fluencia (sigma -> 0 si eps_p -> 0),
    #    por eso solo se dibuja en la zona plástica y no empalma con la recta elástica.
    hol = K_H*epsp[plast]**n
    # 4) Ludwik: sigma = Y + K_L * eps_p^n_L  (arranca en Y cuando eps_p = 0)
    lud = np.where(eps <= eps_y, E*eps, Y + K_L*epsp**n_L)

    ax.plot(eps, epp, lw=2.0, label='elástico–perfectamente plástico')
    ax.plot(eps, lin, lw=2.0, label=r'elástico–lineal con endurec. ($H$)')
    ax.plot(eps[plast], hol, lw=2.0, label=r'Hollomon  $\sigma=K_H\varepsilon_p^{\,n}$')
    ax.plot(eps, lud, lw=2.0, label=r'Ludwik  $\sigma=\sigma_Y+K_L\varepsilon_p^{\,n_L}$')

    # 5) Ramberg-Osgood: eps = sigma/E + 0.002 (sigma/Y)^m -> se construye sobre el eje sigma
    sig_ro = np.linspace(1e-6, max(Y*1.8, lud.max()), 300)
    eps_ro = sig_ro/E + OFFSET*(sig_ro/Y)**m
    ro_mask = eps_ro <= eps_max
    ax.plot(eps_ro[ro_mask], sig_ro[ro_mask], lw=2.0, ls='--',
            label=r'Ramberg–Osgood  ($m$)')

    ax.axhline(Y, color='0.7', lw=0.8, ls=':')
    ax.plot(eps_y, Y, 'ko', ms=5)
    ax.annotate(r'$Y=%.0f$' % Y, (eps_y, Y), textcoords='offset points',
                xytext=(8, -14), fontsize=11)

    hol_max = hol.max() if hol.size else 0.0
    fix_limits(_lim3, ax, (0, eps_max),
               (0, max(lin.max(), hol_max, lud.max(), Y*1.6)*1.05))
    ax.set_xlabel(r'deformación  $\varepsilon$  (-)')
    ax.set_ylabel(r'tensión  $\sigma$  (MPa)')
    ax.grid(alpha=0.3); ax.legend(loc='lower right', fontsize=9)
    ax.set_title(r'Leyes empíricas tensión–deformación (mismos $E$, $Y$)', fontsize=12)
    fig3.canvas.draw_idle()


panel(plot_laws, dict(
    E_GPa=FloatSlider(value=200, min=50, max=400, step=10,
                      description='E (GPa)', style={'description_width':'initial'}),
    Y=FloatSlider(value=250, min=50, max=600, step=10,
                  description='límite elást. Y (MPa)', style={'description_width':'initial'}),
    H=FloatSlider(value=1500, min=0, max=8000, step=100,
                  description='endurec. H (MPa)', style={'description_width':'initial'}),
    K_H=FloatSlider(value=650, min=200, max=1500, step=25,
                    description='Hollomon  K (MPa)', style={'description_width':'initial'}),
    n=FloatSlider(value=0.15, min=0.02, max=0.5, step=0.01,
                  description='Hollomon  n', style={'description_width':'initial'}),
    K_L=FloatSlider(value=500, min=100, max=2000, step=50,
                    description='Ludwik  K (MPa)', style={'description_width':'initial'}),
    n_L=FloatSlider(value=0.15, min=0.02, max=0.5, step=0.01,
                    description='Ludwik  n', style={'description_width':'initial'}),
    m=FloatSlider(value=8.0, min=2.0, max=25.0, step=1.0,
                  description='exp. R–O  m', style={'description_width':'initial'}),
), fig3)

## Resumen — relaciones clave

| Magnitud | Ingenieril | Verdadera |
|---|---|---|
| tensión | $S=F/A_0$ | $\sigma=F/A=S(1+e)$ |
| deformación | $e=\Delta L/L_0$ | $\varepsilon=\ln(1+e)$ |

* Válidas mientras la deformación es **uniforme** (hasta la estricción); se supone
  incompresibilidad plástica.
* **Límite elástico** $S_Y$: a partir del desfase de deformación plástica del $0{,}2\%$.
* **Inestabilidad (Considère):** $\dfrac{\mathrm d\sigma}{\mathrm d\varepsilon}=\sigma$;
  para Hollomon $\varepsilon_{\text{estr}}=n$ y $S_{\max}$ es la resistencia a tracción (UTS).
* Esta curva uniaxial es la que calibra los criterios de plastificación multiaxiales (Tresca,
  Von Mises) y las leyes de endurecimiento de las secciones siguientes.